# D-FINE-M (Query-Based Detector)

Fine-tunes D-FINE-M (pretrained on Objects365 and COCO, Hugging Face `transformers`) on VisDrone and evaluates it
with the shared COCO protocol (sequence-held-out validation split, every 3rd frame, ignored regions treated as "don't care").

Sections:
1. Setup
2. Configuration
3. Training (resumes automatically from the last saved epoch)
4. Training curves
5. Validation metrics
6. Kaggle submission

## 1. Setup

Set `RUNTIME` to `"colab"` (checkpoints go to Google Drive) or `"local"` (checkpoints go to `runs/`).
On Colab, upload `kaggle.json` to the session first so the dataset can be downloaded.

In [ ]:
RUNTIME = "local"  # "colab" or "local"
REPO_URL = "https://github.com/rudkoLA/ai-hw-2-rudko-omelianchuk.git"

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

if RUNTIME == "colab":
    repo_dir = Path("/content/ai-hw-2-rudko-omelianchuk")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_dir)], check=True)
    os.chdir(repo_dir)
    uploaded_key = Path("/content/kaggle.json")
    if uploaded_key.exists():
        kaggle_key = Path.home() / ".kaggle" / "kaggle.json"
        kaggle_key.parent.mkdir(parents=True, exist_ok=True)
        kaggle_key.write_bytes(uploaded_key.read_bytes())
        kaggle_key.chmod(0o600)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "transformers", "ultralytics", "pycocotools", "kaggle"],
        check=True,
    )

print(f"Working directory: {Path.cwd()}")

In [ ]:
import pandas as pd

from src.dfine import evaluate_dfine, load_dfine, predict_frames, train_dfine
from src.setup_utils import ensure_dataset, resolve_storage, setup_environment
from src.submission import build_submission, load_test_frames, submit_to_kaggle, validate_submission
from src.ultralytics_utils import plot_training_curves

env = setup_environment(seed=42, mount_gdrive=False)
device = env["device"]
dataset_dir, coco_train, coco_val, class_names = ensure_dataset()
del coco_train, coco_val  # the datasets load their own annotations
storage = resolve_storage(RUNTIME, "dfine")

## 2. Configuration

All tunable parameters. `model_config` overrides fields of the Hugging Face `DFineConfig`
(any other `DFineConfig` field can be added there).
Give every experiment its own `run_name`: an existing run with the same name is resumed (or skipped if finished).

`QUICK_TEST = True` trains on a 10% sample of every 15th frame for 10 epochs, for fast comparisons
between settings; its runs get a `_quick` suffix. Use `QUICK_TEST = False` for the final run.

In [ ]:
CFG = {
    "run_name": "dfine_m_1024",
    "checkpoint": "ustc-community/dfine-medium-obj2coco",  # Objects365 + COCO pretrained
    "class_names": class_names,
    "train_stride": 3,             # every n-th frame of each training sequence
    "val_stride": 3,               # every n-th frame of each validation sequence
    "fraction": 1.0,               # fraction of the selected training frames to use
    # Schedule
    "imgsz": 1024,
    "epochs": 20,
    "batch": 4,
    "patience": 10,                # early stopping (epochs without val mAP improvement)
    "lr0": 1e-4,
    "backbone_lr_mult": 0.1,       # backbone LR = lr0 * backbone_lr_mult
    "weight_decay": 1e-4,
    "warmup_iters": 500,
    "lr_min_ratio": 0.05,          # cosine decay to lr0 * lr_min_ratio
    "clip_grad_norm": 0.1,         # 0 disables gradient clipping
    "ema_decay": 0.9999,           # 0 disables the weight EMA
    "ema_warmups": 1000,
    "amp": True,                   # mixed precision (CUDA only)
    # Model (DFineConfig fields)
    "model_config": {
        "num_queries": 300,
        "num_denoising": 100,      # 0 disables denoising training
        "matcher_class_cost": 2.0,
        "matcher_bbox_cost": 5.0,
        "matcher_giou_cost": 2.0,
        "weight_loss_vfl": 1.0,
        "weight_loss_bbox": 5.0,
        "weight_loss_giou": 2.0,
        "weight_loss_fgl": 0.15,   # fine-grained localization loss
        "weight_loss_ddf": 1.5,    # decoupled distillation focal loss (self-distillation)
    },
    # Data
    "mask_ignore": False,          # True: fill ignored regions of training images with gray
    # Inference
    "max_det": 500,
    # Misc
    "workers": 4,
    "seed": 42,
}

QUICK_TEST = False
if QUICK_TEST:
    CFG.update({
        "run_name": CFG["run_name"] + "_quick",
        "train_stride": 15,
        "fraction": 0.1,
        "epochs": 10,
        "patience": 3,
    })

## 3. Training

Every epoch the EMA weights are evaluated on the validation frames; `last.pt`, `results.csv` and (on a new best
mAP@[.5:.95]) `best/` are written to `storage/<run_name>`.
Re-running this cell after an interruption resumes from `last.pt`; a finished run is not retrained.

In [ ]:
run_dir = storage / CFG["run_name"]
train_dfine(CFG, dataset_dir, run_dir, device)

## 4. Training curves

Per-epoch validation mAP and train / validation losses (each loss type summed over decoder layers,
encoder and denoising terms).

In [ ]:
history = plot_training_curves(run_dir)

## 5. Validation metrics

The best weights are evaluated on the validation frames with `evaluate_coco_metrics`
(up to 500 detections per image, no NMS). Results are saved to `storage/<run_name>/metrics.json`.

In [ ]:
best_model = load_dfine(run_dir / "best", device)
metrics = evaluate_dfine(
    best_model,
    dataset_dir,
    val_stride=CFG["val_stride"],
    imgsz=CFG["imgsz"],
    batch=CFG["batch"],
    max_det=CFG["max_det"],
    device=device,
    run_dir=run_dir,
    workers=CFG["workers"],
    amp=CFG["amp"],
)
pd.Series(metrics, name=CFG["run_name"]).round(4).to_frame()

## 6. Kaggle submission

Predicts all test frames with the best weights and writes `storage/<run_name>/submission.csv`
(boxes in original pixel coordinates, at most 500 detections per image, empty rows for frames without detections).

In [ ]:
test_frames = load_test_frames(dataset_dir)
test_records, _ = predict_frames(
    best_model,
    test_frames,
    image_root=dataset_dir,
    imgsz=CFG["imgsz"],
    batch=CFG["batch"],
    max_det=CFG["max_det"],
    device=device,
    workers=CFG["workers"],
    amp=CFG["amp"],
)

submission_path = run_dir / "submission.csv"
build_submission(test_records, test_frames, submission_path)
report = validate_submission(submission_path, test_frames)
report

Set `SUBMIT = True` to upload the file. Kaggle credentials are read from `~/.kaggle/kaggle.json`
or from the `KAGGLE_API_TOKEN` environment variable.

In [ ]:
SUBMIT = False
SUBMISSION_MESSAGE = f"D-FINE-M {CFG['run_name']}"

if SUBMIT:
    assert report["is_valid"], report["errors"]
    submit_to_kaggle(submission_path, SUBMISSION_MESSAGE)